# Task 2 — Train/Test Split

**Saiket Systems | Machine Learning Internship**

Objective: create representative 80% training and 20% testing datasets. A stratified split preserves the churn class ratio.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Find the project root even if this notebook is opened from its own task folder.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "Dataset").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the Dataset folder. Open this notebook inside Saiket_ML_Internship.")
DATA_PATH = ROOT / "Dataset" / "Telco_Customer_Churn_Dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset path:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Dataset path: C:\Users\HP\Desktop\Saiket_ML_Internship\Dataset\Telco_Customer_Churn_Dataset.csv
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Clean and prepare features

All preprocessing is fitted/defined consistently for this task. `customerID` is excluded because it is an identifier, not a customer-behavior feature.

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(r"^\\s*$", np.nan, regex=True), errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
y = df["Churn"].map({"No": 0, "Yes": 1})
X = pd.get_dummies(df.drop(columns=["Churn", "customerID"]), drop_first=True, dtype=int)
print("X shape:", X.shape, "| y shape:", y.shape)


X shape: (7043, 30) | y shape: (7043,)


## 2. Split into 80% train and 20% test

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train):,} ({len(X_train)/len(X):.0%})")
print(f"Testing rows:  {len(X_test):,} ({len(X_test)/len(X):.0%})")
print("\\nChurn share:")
display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index()
}).rename(index={0:"No churn (0)", 1:"Churn (1)"}))


Training rows: 5,634 (80%)
Testing rows:  1,409 (20%)
\nChurn share:


,train,test
Churn,,
No churn (0),0.734647,0.734564
Churn (1),0.265353,0.265436


## 3. Verify the split

In [4]:
assert set(X_train.columns) == set(X_test.columns)
assert len(X_train) + len(X_test) == len(X)
print("Feature columns match:", X_train.columns.equals(X_test.columns))
print("Split validation passed.")


Feature columns match: True
Split validation passed.


## 4. Summary

`X_train`, `X_test`, `y_train`, and `y_test` are available in this notebook for subsequent modelling. `random_state=42` makes the split reproducible.